# 1장 3강: 선형 변환과 신경망 레이어 — 실습문제

## 실습 목표

- 같은 위치끼리 곱해 더하여 선형 변환을 계산합니다.
- 편향을 더하는 아핀 변환과 선형 변환의 차이를 확인합니다.
- 가중치와 편향을 각각 바꿀 때 점수가 어떻게 달라지는지 확인합니다.
- ReLU의 규칙을 적용하고, 두 레이어의 계산을 순서대로 따라갑니다.

## 실습 환경 / 데이터

- Python, NumPy를 사용합니다. 외부 데이터 파일은 필요하지 않습니다.
- 제공된 「1장 3강: 선형 변환과 신경망 레이어」의 예제와 연습문제에 있는 숫자·방법을 사용합니다.
- 새로운 공식을 유도하거나 모델을 학습시키는 과제는 없습니다.

## 실습 구성

| 구분 | 내용 | 교안 근거 |
|---|---|---|
| 필수 1 / 문제 1-1 | 선형·아핀 변환과 0벡터 | 1절의 x=[3,2] 예제 |
| 필수 1 / 문제 1-2 | 가중치와 편향을 따로 변경 | 2절·연습문제 2 |
| 필수 2 / 문제 2-1 | ReLU와 한 레이어 계산 | 3절의 고객 3명 예제 |
| 필수 2 / 문제 2-2 | 은닉층에서 출력층으로 전달 | 3절의 전체 순전파 예제 |
| 과제 / 문제 3-1 | 선형·아핀 변환 반복 | 교안 연습문제 1 |
| 과제 / 문제 3-2 | 두 레이어 순전파 반복 | 교안 연습문제 3 |

필수 실습 4문제와 기초 과제 2문제입니다. 과제도 필수 실습에서 사용한 계산 순서를 그대로 적용합니다.

## 진행 안내

1. 위에서 아래로 읽고, 먼저 NumPy를 불러오는 셀을 실행합니다.
2. 각 문제의 숫자와 작성 도움말을 보고 코드를 한 줄씩 작성합니다.
3. 출력값을 확인한 뒤 답변 작성란에 계산 과정과 답을 적습니다.
4. 수식은 먼저 말로 읽고, 곱셈 → 덧셈 → 편향 → ReLU 순서를 확인합니다. ReLU는 적용하라고 한 곳에서만 사용합니다.
5. 변수는 각 문제에서 다시 만듭니다. 앞 문제의 입력값이나 가중치를 그대로 쓰지 않도록 주의하세요.

## 처음 보는 용어와 기호

| 용어·기호 | 쉬운 뜻 |
|---|---|
| 입력 / 출력 | 계산에 넣는 숫자 / 계산해서 얻는 숫자 |
| 벡터 / 행렬 | 숫자를 한 줄로 나열한 것 / 숫자를 행과 열로 놓은 것 |
| 행 / 열 | 가로로 읽는 줄 / 세로로 읽는 줄 |
| 특성 | 고객을 나타내는 한 가지 정보. 예: 방문 횟수 |
| x / X | 한 고객의 입력 / 여러 고객의 입력을 모은 배열 |
| w / W | 입력값에 곱할 가중치. W의 각 열은 출력 하나에 사용할 가중치 |
| b / bias | 곱하고 더한 결과에 마지막으로 더할 편향 |
| 파라미터 | 이 강의에서는 가중치와 편향을 가리키는 말 |
| 뉴런 / 레이어 | 입력으로 출력 하나를 만드는 계산 단위 / 뉴런들을 모은 계산 단계 |
| 은닉층 / 출력층 | 입력과 최종 결과 사이의 계산 단계 / 최종 결과를 만드는 단계 |
| Z, Z1 | 가중치를 곱해 더하고 편향까지 더한 값 |
| A, H | ReLU를 적용한 값. 고객 이름 A와 코드 변수 A는 문맥으로 구분 |
| Y | 최종 출력값을 모은 배열 |
| Shape | 배열의 모양. (3,2)는 3행 2열, (2,)는 숫자 2개인 1차원 배열 |

`xW`는 수식에서 x와 W의 행렬곱을 뜻하며, 코드에서는 `x @ W`로 씁니다. `xW+b`는 그 결과에 편향 b를 더하라는 뜻입니다. `T(x)`는 ‘입력 x에 T라는 계산을 적용한 결과’로 읽습니다.

In [ ]:
# 순서
# 1. 입력층 -> 정보 전달
# 2. 첫번째 은닉층 -> 뉴런 n개가 각 가중합과 편향 계산
# 3. 첫번째 은닉층의 ReLU -> 음수 결과를 0으로 변경
# 4. 두번째 은닉층 -> 첫번째 은닉층의 ReLU 결과 전체를 입력으로 받아,
#                   두번째 은닉층의 뉴런 m개가 각각 가중합과 편향 계산
# 5. 두번째 은닉층의 ReLU -> 음수 결과를 0으로 변경
# 6. 출력층 -> 마지막 가중합과 편향을 계산
#            (회귀는 활성화 없음 / 이진분류는 시그모이드 / 다중분류는 softmax)

## 실습 준비

아래 셀을 먼저 실행하세요. NumPy는 배열을 계산하는 도구이며, `np`는 코드에서 NumPy를 짧게 부르는 이름입니다.

In [1]:
import numpy as np

---

## 필수 1. 선형·아핀 변환과 파라미터 이해하기

### 문제 1-1. 선형 변환에 편향을 더해 보기

#### 문제 설명

교안 1절의 예제입니다. 선형 변환은 입력에 가중치 행렬을 곱하는 계산이며, 아핀 변환은 그 결과에 편향을 더하는 계산입니다.

```python
x = np.array([3, 2])
W = np.array([[2, 0], [0, 3]])
bias = np.array([1, -2])
zero = np.array([0, 0])
```

- 선형 변환: `x @ W`
- 아핀 변환: `x @ W + bias`

W의 첫 번째 열은 [2,0], 두 번째 열은 [0,3]입니다. 0벡터는 모든 값이 0인 벡터를 말합니다.

#### 요구사항

1. x, W, bias, zero를 위와 같이 만드세요.
2. `linear_output = x @ W`, `affine_output = x @ W + bias`를 계산하여 출력하세요.
3. `zero @ W`와 `zero @ W + bias`를 출력하세요.
4. 답변 작성란에 두 출력값의 곱셈·덧셈 과정을 적으세요.

#### 작성 도움말

첫 출력은 x의 [3,2]와 W의 첫 열 [2,0]을 같은 순서로 짝지어 `3×2 + 2×0`으로 계산합니다. 둘째 출력도 둘째 열을 사용해 같은 방법으로 계산하세요. 편향은 첫 출력에 1, 둘째 출력에 −2를 각각 더합니다.

#### 해석 질문

**Q1.** 선형 변환의 첫 번째·두 번째 출력은 각각 어떻게 계산되나요?

**Q2.** 편향을 더한 아핀 변환 결과는 무엇인가요?

**Q3.** 0벡터 입력에서 두 변환의 결과는 각각 무엇인가요?

**Q4.** 편향이 [0,0]이라면 두 변환의 결과가 같아지나요?

#### 제출 결과

- 두 변환의 출력과 0벡터의 출력
- 계산 과정과 Q1~Q4 답변

In [ ]:
# 문제 1-1. 선형 변환에 편향을 더해 보기
# 위의 데이터와 작성 도움말을 보고 한 단계씩 코드를 작성하세요.
# 중간 결과를 출력하여 확인하세요.

# 1. x, W, bias, zero를 위와 같이 만드세요.
x = np.array([3, 2]) # 입력
W = np.array([[2, 0], [0, 3]]) # 가중치 조절
bias = np.array([1, -2]) 
zero = np.array([0, 0])

# 2. `linear_output = x @ W`, `affine_output = x @ W + bias`를 계산하여 출력하세요.
linear_output = x @ W # 선형 출력 계산
affine_output = x @ W + bias # 아핀 변환 계산
print("선형 변환 :", linear_output)
print("아핀 변환 :", affine_output)

# 3. `zero @ W`와 `zero @ W + bias`를 출력하세요.
zero_linear = zero @ W
zero_affine = zero @ W + bias
print("zero 선형 변환 :", zero_linear)
print("zero 아핀 변환 :", zero_affine)

# 4. 답변 작성란에 두 출력값의 곱셈·덧셈 과정을 적으세요.

선형 변환 : [6 6]
아핀 변환 : [7 4]
zero 선형 변환 : [0 0]
zero 아핀 변환 : [ 1 -2]


#### 답변 작성란

**Q1. 선형 변환의 첫 번째·두 번째 출력은 각각 어떻게 계산되나요?**

- 첫번째 출력 : 6  
-> 3x2 + 2x0 = 6  
- 두번째 출력 : 6  
-> 2x0 + 2x3 = 6  

**Q2. 편향을 더한 아핀 변환 결과는 무엇인가요?**

- 첫번째 출력 : 7  
-> 3x2 + 2x0 + `1` = 7  
- 두번째 출력 : 4  
2x0 + 2x3 + `(-2)` = 4  

**Q3. 0벡터 입력에서 두 변환의 결과는 각각 무엇인가요?**

- **0벡터 선형 변환 : [0 0]**  
-> 0x2 + 0x0 = 0 , 0x0 + 0x3 = 0  
**0벡터 아핀 변환 : [1 -2]**  
-> 0x2 + 0x0 + `1` = 1 , 0x0 + 0x3 + `(-2)` = -2  
-> 입력이 모두 0이면 가중합은 0이지만 편향은 남는다.

**Q4. 편향이 [0,0]이라면 두 변환의 결과가 같아지나요?**

- 답변: 네  
아핀 변환은 선형 변환에서 편향을 더해주기 때문에 편향이 [0,0]이라면 아핀 변환도 [0 0]으로 바뀜  
-> 선형 변환과 아핀 변환의 결과가 같아짐  
> 가중치 행렬의 열마다 곱해 더하면 선형 변환, 그 결과에 편향을 더하면 아핀 변환이라고 부름

### 문제 1-2. 가중치와 편향을 하나씩 바꾸기

#### 문제 설명

교안 연습문제 2의 데이터입니다. 각 고객의 첫 입력은 방문 횟수, 둘째 입력은 장바구니 추가 횟수입니다.

```python
X = np.array([[4, 2], [1, 5]])
w = np.array([2, -1])
b = 3
```

X의 첫 행 [4,2]는 고객 A, 둘째 행 [1,5]는 고객 B입니다. 고객별 점수는 ‘첫 입력×첫 가중치 + 둘째 입력×둘째 가중치 + 편향’으로 계산합니다.

| 비교 | 가중치 | 편향 |
|---|---|---|
| 기본 | [2,−1] | 3 |
| 가중치만 변경 | [2,−2] | 3 |
| 편향만 변경 | [2,−1] | 5 |

두 변경은 각각 기본 설정에서 출발합니다.

#### 요구사항

1. 위 배열을 만들고 `base_scores = X @ w + b`를 계산하여 출력하세요.
2. `w_new = np.array([2, -2])`로 만들고 `weight_scores = X @ w_new + b`를 계산하세요. 점수와 `weight_scores - base_scores`를 출력하세요.
3. 기본 w를 사용하여 `b_new = 5`, `bias_scores = X @ w + b_new`를 계산하세요. 점수와 `bias_scores - base_scores`를 출력하세요.
4. `zero = np.array([0, 0])`를 만들고 기본 설정인 `zero @ w + b`를 출력하세요.

#### 작성 도움말

변화량은 ‘변경 후 점수 − 기본 점수’입니다. 변화량이 −2라면 점수가 2만큼 줄었다는 뜻입니다. 편향만 바꿀 때는 w_new가 아닌 기본 w를 사용하세요. 한 번에 한 조건만 바꿉니다.

#### 해석 질문

**Q1.** 기본 설정에서 고객 A와 B의 점수는 각각 얼마인가요?

**Q2.** 가중치만 바꾸면 각 점수와 변화량은 얼마인가요?

**Q3.** 편향만 바꾸면 각 점수와 변화량은 얼마인가요?

**Q4.** 기본 설정에 0벡터를 넣으면 점수가 얼마인가요? 왜 그 값이 남나요?

#### 제출 결과

- 기본 점수, 두 변경의 점수와 변화량, 0벡터의 점수
- 계산 과정과 Q1~Q4 답변

In [ ]:
# 문제 1-2. 가중치와 편향을 하나씩 바꾸기
# 위의 데이터와 작성 도움말을 보고 한 단계씩 코드를 작성하세요.
# 중간 결과를 출력하여 확인하세요.

# 1. 위 배열을 만들고 `base_scores = X @ w + b`를 계산하여 출력하세요.
X = np.array([[4, 2],    # 고객 A
              [1, 5]])   # 고객 B
w = np.array([2, -1])    # 가중치 배정 첫 입력에 2, 둘째 입력에 -1
b = 3 # 고객마다 곱셈 결과를 더한 뒤 3을 더함

base_scores = X @ w + b # 각 행의 두 입력과 w의 두 가중치를 짝지어 계산
print("기본 점수 :", base_scores)

# 2. `w_new = np.array([2, -2])`로 만들고 `weight_scores = X @ w_new + b`를 계산하세요. 점수와 `weight_scores - base_scores`를 출력하세요.
w_new = np.array([2, -2]) # 가중치 배정 두번째 가중치만 -2로 수정
weight_scores = X @ w_new + b # 가중치 변화에 따른 변화량 산출
print("가중치 변경 점수 :", weight_scores)
print("가중치 변경 변화량 :", weight_scores - base_scores)

# 3. 기본 w를 사용하여 `b_new = 5`, `bias_scores = X @ w + b_new`를 계산하세요. 점수와 `bias_scores - base_scores`를 출력하세요.
b_new = 5 # 새로운 편향
bias_scores = X @ w + b_new # 기본 가중치로 돌아가 편향만 수정
print("편향 변경 점수 :", bias_scores) 
print("편향 변경 변화량 :", bias_scores - base_scores)

# 4. `zero = np.array([0, 0])`를 만들고 기본 설정인 `zero @ w + b`를 출력하세요.
zero = np.array([0, 0]) # 0벡터 점수 확인
print("0벡터 점수 :", zero @ w + b)


기본 점수 : [9 0]
가중치 변경 점수 : [ 7 -5]
가중치 변경 변화량 : [-2 -5]
편향 변경 점수 : [11  2]
편향 변경 변화량 : [2 2]
0벡터 점수 : 3


#### 답변 작성란

**Q1. 기본 설정에서 고객 A와 B의 점수는 각각 얼마인가요?**

- 답변: [9, 0]  
고객 A : 4 × 2 + 2 × (−1) + 3 
8 − 2 + 3  
-> 9  
고객 B : 1 × 2 + 5 × (−1) + 3  
2 − 5 + 3  
-> 0  

**Q2. 가중치만 바꾸면 각 점수와 변화량은 얼마인가요?**

- 답변: 점수 [7, −5], 변화량 [−2, −5]  
고객 A : 4 × 2 + 2 × (−2) + 3 = 8 − 4 + 3 = 7  
-> 변화량 7 − 9 = −2  
고객 B : 1 × 2 + 5 × (−2) + 3 = 2 − 10 + 3 = −5  
-> 변화량 −5 − 0 = −5  
-> 둘째 가중치가 1만큼 줄었으므로, 장바구니 추가 횟수가 많은 고객 B(5회)가 A(2회)보다 점수가 더 많이 감소  

**Q3. 편향만 바꾸면 각 점수와 변화량은 얼마인가요?**

- 답변: 점수 [11, 2], 변화량 [2, 2]  
고객 A : 4 × 2 + 2 × (−1) + 5 = 8 − 2 + 5 = 11 -> 변화량 11 − 9 = 2  
고객 B : 1 × 2 + 5 × (−1) + 5 = 2 − 5 + 5 = 2 -> 변화량 2 − 0 = 2  
-> 편향이 3에서 5로 2만큼 커졌고, 가중합은 그대로라서 두 점수에 똑같이 2를 더함

**Q4. 기본 설정에 0벡터를 넣으면 점수가 얼마인가요? 왜 그 값이 남나요?**

- 답변: 3  
0 × 2 + 0 × (−1) + 3 = 0 + 0 + 3 = 3  
-> 입력이 모두 0이기 때문에 가중치를 곱한 값은 0이고, 마지막에 편향만 남음  

---

## 필수 2. ReLU와 신경망 순전파

### 문제 2-1. ReLU와 한 레이어를 순서대로 계산하기

#### 문제 설명

교안 3절의 예제입니다. 먼저 ReLU의 규칙을 연습한 뒤 레이어 하나를 계산합니다.

**ReLU(렐루)의 규칙:** 양수는 그대로 두고, 음수는 0으로 바꿉니다. 0은 0으로 둡니다.

```python
def relu(z):
    return np.maximum(0, z)

test_z = np.array([-3, -1, 0, 2, 5])
X = np.array([[3, 2], [1, 4], [0, 2]])
W = np.array([[1, -2], [2, 1]])
b = np.array([-4, 1])
```

X의 세 행은 고객 A, B, C입니다. W의 두 열은 뉴런 두 개의 가중치입니다. 첫 뉴런의 가중치는 [1,2], 둘째 뉴런의 가중치는 [−2,1]입니다. 편향은 각각 −4와 1입니다.

이번 계산은 **가중치 적용 → 편향 덧셈 → ReLU**의 세 단계입니다.

#### 요구사항

1. 위 코드를 입력하고 `relu(test_z)`를 출력하세요.
2. `weighted = X @ W`를 계산하여 출력하세요.
3. `Z = weighted + b`를 계산하여 출력하세요.
4. `A = relu(Z)`를 계산하여 출력하세요.
5. `Z.shape`와 `A.shape`를 출력하고, 고객 A의 두 뉴런 계산을 손으로 확인하세요.

#### 작성 도움말

`def relu(z):`와 그 아래 `return` 줄은 위 예제를 그대로 사용하세요. return 앞의 들여쓰기도 유지하세요. 고객 A의 첫 뉴런은 `3×1 + 2×2`를 구한 뒤 −4를 더합니다. 마지막에 그 값과 0 중 큰 값을 고릅니다.

#### 해석 질문

**Q1.** ReLU는 [−3,−1,0,2,5]를 어떻게 바꾸나요?

**Q2.** 고객 A의 가중합, 편향을 더한 값 Z, ReLU 결과 A는 각각 무엇인가요?

**Q3.** 고객 B와 C의 Z와 ReLU 결과는 각각 무엇인가요?

**Q4.** Z와 A의 Shape (3,2)는 무엇을 뜻하나요? 음수를 0으로 바꾸면 배열의 크기도 줄어드나요?

#### 제출 결과

- ReLU 연습 결과, weighted·Z·A의 출력, Z·A의 Shape
- 고객 A의 계산 과정과 Q1~Q4 답변

In [31]:
# 문제 2-1. ReLU와 한 레이어를 순서대로 계산하기
# 위의 데이터와 작성 도움말을 보고 한 단계씩 코드를 작성하세요.
# 중간 결과를 출력하여 확인하세요.

# 1. 위 코드를 입력하고 `relu(test_z)`를 출력하세요.'
def relu(z):
    return np.maximum(0, z) # z와 0값을 비교하여 큰 값을 return

test_z = np.array([-3, -1, 0, 2, 5]) # ex) relu 적용시 : 0, 0, 0, 2, 5
X = np.array([
    [3, 2],    # 고객 A
    [1, 4],    # 고객 B
    [0, 2]     # 고객 C
    ])

W = np.array([
    [1, -2],
    [2,  1] # w의 첫 열 [1,2] <-첫 뉴런
    ])      # w의 둘째 열 [-2, 1]은 둘째 뉴런에 사용
            # 두 뉴런은 같은 고객의 입력 2개를 각각 받아 서로 다른 가중치로 계산한다.

b = np.array([-4, 1]) # 편향이 적용되는 위치 확인
print("x shape :", x.shape)
print("w shape :", w.shape)
print("b shape :", b.shape)

print("relu(test_z) :", relu(test_z))


# 2. `weighted = X @ W`를 계산하여 출력하세요.
weighted = X @ W
print("\n== weighted ==")
print(weighted)

# 3. `Z = weighted + b`를 계산하여 출력하세요.
Z = weighted + b
print("\n== Z ==")
print(Z)

# 4. `A = relu(Z)`를 계산하여 출력하세요.
A = relu(Z)
print("\n== A ==")
print(A)

# 5. `Z.shape`와 `A.shape`를 출력하고, 고객 A의 두 뉴런 계산을 손으로 확인하세요.
print("\nZ shape :", Z.shape)
print("A shape :", A.shape)


x shape : (2,)
w shape : (2,)
b shape : (2,)
relu(test_z) : [0 0 0 2 5]

== weighted ==
[[ 7 -4]
 [ 9  2]
 [ 4  2]]

== Z ==
[[ 3 -3]
 [ 5  3]
 [ 0  3]]

== A ==
[[3 0]
 [5 3]
 [0 3]]

Z shape : (3, 2)
A shape : (3, 2)


#### 답변 작성란

**Q1. ReLU는 [−3,−1,0,2,5]를 어떻게 바꾸나요?**

- 답변: [0, 0, 0, 2, 5]  
−3, −1 : 음수이므로 0  
0 : 0은 그대로 0  
2, 5 : 양수이므로 그대로  

**Q2. 고객 A의 가중합, 편향을 더한 값 Z, ReLU 결과 A는 각각 무엇인가요?**

- 답변: 가중합 [7, −4], Z [3, −3], ReLU 결과 [3, 0]  
가중합 : 3 × 1 + 2 × 2 = 7, 3 × (−2) + 2 × 1 = −4 -> [7, −4]  
Z : 7 + (−4) = 3, −4 + 1 = −3 -> [3, −3]  
ReLU : max(0, 3) = 3, max(0, −3) = 0 -> [3, 0]  

**Q3. 고객 B와 C의 Z와 ReLU 결과는 각각 무엇인가요?**

- 답변:  
고객 B  
Z : 1 × 1 + 4 × 2 + (−4) = 5, 1 × (−2) + 4 × 1 + 1 = 3 -> [5, 3]  
ReLU : 둘 다 양수라 그대로 -> [5, 3]  
고객 C  
Z : 0 × 1 + 2 × 2 + (−4) = 0, 0 × (−2) + 2 × 1 + 1 = 3 -> [0, 3]  
ReLU : 둘 다 양수라 그대로 -> [0, 3]  

**Q4. Z와 A의 Shape (3,2)는 무엇을 뜻하나요? 음수를 0으로 바꾸면 배열의 크기도 줄어드나요?**

- 답변: 3은 고객 수(A, B, C), 2는 뉴런 수(고객 1명당 출력 2개)  
Z와 A 모두 (3,2)의 shape를 가짐  
ReLU는 음수를 0으로 바꿔도 배열의 크기는 줄어들지 않음  
-> ReLU는 각 칸의 값만 바꾸고 칸을 지우지 않으므로, Z와 A의 Shape는 (3, 2)로 같음

### 문제 2-2. 은닉층의 결과를 출력층에 전달하기

#### 문제 설명

교안 3절의 전체 순전파 예제입니다. 순전파란 입력부터 출력까지 앞 단계의 결과를 다음 단계로 전달하며 계산하는 과정입니다.

```python
X = np.array([[3, 2], [1, 4], [0, 2]])
W1 = np.array([[1, -2], [2, 1]])
b1 = np.array([-4, 1])
W2 = np.array([[2], [-1]])
b2 = np.array([1])
```

첫 단계인 은닉층은 문제 2-1과 같습니다. 은닉층의 뉴런은 2개이고, 그 뒤에 뉴런 1개인 출력층을 붙입니다. 아래 세 줄을 위에서부터 계산하세요.

```python
Z1 = X @ W1 + b1
H = relu(Z1)
Y = H @ W2 + b2
```

Z1은 은닉층에서 편향까지 더한 값, H는 ReLU를 적용한 값, Y는 최종 점수입니다. 은닉층에는 ReLU를 적용하고, 이 교안 예제의 출력층에는 별도 활성화 함수를 적용하지 않습니다.

#### 요구사항

1. 문제 2-1의 relu 함수를 그대로 작성하고, 위 입력과 파라미터를 만드세요.
2. Z1, H, Y를 제시된 순서대로 계산하여 출력하세요.
3. `H.shape`와 `Y.shape`를 출력하세요.
4. 고객별 H의 두 숫자를 사용해 최종 점수를 손으로 계산하세요.

#### 작성 도움말

W2의 첫 값은 2, 둘째 값은 −1입니다. 따라서 H의 한 행이 [h₁,h₂]이면 최종 점수는 `h₁×2 + h₂×(−1) + 1`입니다. h₁과 h₂는 각각 그 행의 첫째·둘째 숫자를 가리킵니다. 출력층에는 X나 Z1 대신 H를 넣으세요.

#### 해석 질문

**Q1.** Z1과 H는 어떻게 다른가요? 출력층의 입력으로 어느 것을 사용하나요?

**Q2.** 고객 A·B·C의 최종 점수는 각각 얼마인가요? 계산 과정을 적으세요.

**Q3.** H의 Shape (3,2)와 Y의 Shape (3,1)는 각각 무엇을 뜻하나요?

**Q4.** 고객 C의 최종 점수가 음수로 나와도 되는 이유는 무엇인가요?

#### 제출 결과

- Z1·H·Y 출력과 H·Y의 Shape
- 고객별 최종 계산 과정과 Q1~Q4 답변

In [35]:
# 문제 2-2. 은닉층의 결과를 출력층에 전달하기
# 위의 데이터와 작성 도움말을 보고 한 단계씩 코드를 작성하세요.
# 중간 결과를 출력하여 확인하세요.

# 1. 문제 2-1의 relu 함수를 그대로 작성하고, 위 입력과 파라미터를 만드세요.
def relu(z):
    return np.maximum(0, z)

X = np.array([
    [3, 2],  # 고객 A
    [1, 4],  # 고객 B
    [0, 2]   # 고객 C
    ])

W1 = np.array([
    [1, -2],
    [2,  1]
    ])

b1 = np.array([-4, 1])

W2 = np.array([
    [2],
    [-1]
    ])

b2 = np.array([1])


# 2. Z1, H, Y를 제시된 순서대로 계산하여 출력하세요.
Z1 = X @ W1 + b1
print("== Z1 ==")
print(Z1)

H = relu(Z1)
print("\n==은닉층 H 출력==")
print(H)

Y = H @ W2 + b2
print("\n==최중 Y 출력==")
print(Y)


# 3. `H.shape`와 `Y.shape`를 출력하세요.
print("\nH shape :", H.shape)
print("Y shape :", Y.shape)

== Z1 ==
[[ 3 -3]
 [ 5  3]
 [ 0  3]]

==은닉층 H 출력==
[[3 0]
 [5 3]
 [0 3]]

==최중 Y 출력==
[[ 7]
 [ 8]
 [-2]]

H shape : (3, 2)
Y shape : (3, 1)


#### 답변 작성란

**Q1. Z1과 H는 어떻게 다른가요? 출력층의 입력으로 어느 것을 사용하나요?**

- 답변: Z1은 ReLU 적용 전, H는 ReLU 적용 후  
Z1 : 가중치를 곱하고 편향까지 더한 값 (음수 포함 가능) -> [[3, −3], [5, 3], [0, 3]]  
H : Z1에 ReLU를 적용해 음수를 0으로 바꾼 값 -> [[3, 0], [5, 3], [0, 3]]  
-> 출력층의 입력으로는 **H**를 사용 (은닉층의 최종 출력이 H이기 때문)  

**Q2. 고객 A·B·C의 최종 점수는 각각 얼마인가요? 계산 과정을 적으세요.**

- 답변: [7, 8, −2]  
고객 A : H = [3, 0] -> 3 × 2 + 0 × (−1) + 1 = 6 + 0 + 1 = 7  
고객 B : H = [5, 3] -> 5 × 2 + 3 × (−1) + 1 = 10 − 3 + 1 = 8  
고객 C : H = [0, 3] -> 0 × 2 + 3 × (−1) + 1 = 0 − 3 + 1 = −2  

**Q3. H의 Shape (3,2)와 Y의 Shape (3,1)는 각각 무엇을 뜻하나요?**

- 답변:  
H (3, 2) : 고객 3명 × 은닉층 뉴런 2개 -> 고객 1명당 은닉층 출력 2개  
Y (3, 1) : 고객 3명 × 출력층 뉴런 1개 -> 고객 1명당 최종 점수 1개  
-> 고객 수 3은 그대로 유지되고, 두 번째 숫자만 각 층의 뉴런 수에 따라 바뀜  
-> Y의 (3,1)은 고객 3명의 최종 출력이 1개씩 있다는 뜻

**Q4. 고객 C의 최종 점수가 음수로 나와도 되는 이유는 무엇인가요?**

- 답변: 출력층에는 활성화 함수를 적용하지 않았기 때문  
ReLU는 은닉층에만 적용했으므로 음수를 0으로 바꾸는 과정은 은닉층에서만 일어남  
-> 출력층은 H @ W2 + b2의 아핀 변환 결과를 그대로 내보내므로 음수도 나올 수 있음  

---

## 과제. 필수 실습과 같은 방법 적용하기

### 문제 3-1. 선형 변환과 아핀 변환 다시 계산하기

#### 문제 설명

교안 연습문제 1입니다. 문제 1-1과 같은 방법을 사용하세요.

```python
x = np.array([2, 3])
W = np.array([[2, 0], [1, 3]])
bias = np.array([1, -2])
zero = np.array([0, 0])
```

W의 첫 열은 [2,1], 둘째 열은 [0,3]입니다. 이번에는 첫 열의 두 번째 가중치가 0이 아닌 1이라는 점을 확인하세요.

#### 요구사항

1. 위 배열을 만들고 `linear_output = x @ W`를 계산하여 출력하세요.
2. `affine_output = x @ W + bias`를 계산하여 출력하세요.
3. `zero @ W`와 `zero @ W + bias`를 출력하세요.
4. 각 출력값에 사용한 곱셈과 덧셈을 답변 작성란에 적으세요.

#### 작성 도움말

첫 출력은 `2×2 + 3×1`입니다. 둘째 출력은 둘째 열 [0,3]을 이용하세요. 선형 변환을 모두 구한 뒤 각 위치의 편향을 더합니다. 새로운 함수나 공식을 찾을 필요 없이 문제 1-1의 순서대로 계산하세요.

#### 해석 질문

**Q1.** 선형 변환의 첫 번째·두 번째 출력은 각각 어떻게 계산되나요?

**Q2.** 편향을 더한 아핀 변환 결과는 무엇인가요?

**Q3.** 0벡터 입력에서 선형 변환과 아핀 변환 결과는 각각 무엇인가요?

#### 제출 결과

- 선형·아핀 변환 및 0벡터의 출력
- 계산 과정과 Q1~Q3 답변

In [15]:
# 문제 3-1. 선형 변환과 아핀 변환 다시 계산하기
# 위의 데이터와 작성 도움말을 보고 한 단계씩 코드를 작성하세요.
# 중간 결과를 출력하여 확인하세요.


# 1. 위 배열을 만들고 `linear_output = x @ W`를 계산하여 출력하세요.
x = np.array([2, 3])

W = np.array([
    [2, 0], 
    [1, 3]
    ])

bias = np.array([1, -2])

zero = np.array([0, 0])

linear_output = x @ W
print("linear_output :", linear_output)

# 2. `affine_output = x @ W + bias`를 계산하여 출력하세요.
affine_output = x @ W + bias
print("affine_output :", affine_output)


# 3. `zero @ W`와 `zero @ W + bias`를 출력하세요.
zero_linear = zero @ W
zero_affine = zero @ W + bias

print("0벡터 선형 변환 :", zero_linear)
print("0벡터 아핀 변환 :", zero_affine)

# 4. 각 출력값에 사용한 곱셈과 덧셈을 답변 작성란에 적으세요.

linear_output : [7 9]
affine_output : [8 7]
0벡터 선형 변환 : [0 0]
0벡터 아핀 변환 : [ 1 -2]


#### 답변 작성란

**Q1. 선형 변환의 첫 번째·두 번째 출력은 각각 어떻게 계산되나요?**

- 첫 번째 출력: 7  
-> 2×2 + 3×1 = 4 + 3 = 7  
- 두 번째 출력 : 9  
->  2×0 + 3×3 = 0 + 9 = 9

**Q2. 편향을 더한 아핀 변환 결과는 무엇인가요?**

- 첫 번째 출력 : 8  
7 + 1 = 8  
- 두 번째 출력 : 7  
9 + (-2) = 7  


**Q3. 0벡터 입력에서 선형 변환과 아핀 변환 결과는 각각 무엇인가요?**

- 선형 변환: 0×2 + 0×1 = **0**, 0×0 + 0×3 = **0** -> **[0, 0]**
- 아핀 변환: 0×2 + 0×1 + 1 = **1**, 0×0 + 0×3 + **(-2)** = -2 -> **[1, -2]**

### 문제 3-2. 같은 순서로 두 레이어 계산하기

#### 문제 설명

교안 연습문제 3입니다. 문제 2-2와 같은 순서를 적용합니다.

```python
X = np.array([[2, 1], [1, 3]])
W1 = np.array([[1, -1], [2, 1]])
b1 = np.array([-3, 0])
W2 = np.array([[2], [-1]])
b2 = np.array([1])
```

첫 행은 고객 A, 둘째 행은 고객 B입니다. 은닉층에서 ReLU를 적용하고, 출력층에는 별도 활성화 함수를 적용하지 않습니다.

#### 요구사항

1. 문제 2-1의 relu 함수를 작성하고 위 배열을 만드세요.
2. `Z1 = X @ W1 + b1`을 계산하여 출력하세요.
3. `H = relu(Z1)`을 계산하여 출력하세요.
4. `Y = H @ W2 + b2`와 `Y.shape`를 출력하세요.
5. 아래 질문 순서대로 두 고객의 계산 과정을 적으세요.

#### 작성 도움말

은닉층 첫 뉴런은 첫 열 [1,2]를 사용하고 편향 −3을 더합니다. 둘째 뉴런은 둘째 열 [−1,1]을 사용하고 편향 0을 더합니다. 그 다음 ReLU를 적용하세요. 출력층에서는 H의 첫 값×2 + 둘째 값×(−1) + 1을 계산합니다.

#### 해석 질문

**Q1.** 고객 A의 은닉층 Z1과 ReLU 결과 H는 각각 무엇인가요?

**Q2.** 고객 B의 은닉층 Z1과 ReLU 결과 H는 각각 무엇인가요?

**Q3.** 고객 A와 B의 최종 점수는 각각 얼마인가요? H의 값을 사용해 계산하세요.

**Q4.** 최종 출력의 Shape는 무엇이고, 각 숫자는 무엇을 뜻하나요?

#### 제출 결과

- Z1·H·Y 출력과 Y의 Shape
- 고객별 단계별 계산과 Q1~Q4 답변

In [27]:
# 문제 3-2. 같은 순서로 두 레이어 계산하기
# 위의 데이터와 작성 도움말을 보고 한 단계씩 코드를 작성하세요.
# 중간 결과를 출력하여 확인하세요.

# 1. 문제 2-1의 relu 함수를 작성하고 위 배열을 만드세요.
def relu(z):
    return np.maximum(0,z)

X = np.array([
    [2, 1], 
    [1, 3]
    ])

W1 = np.array([
    [1, -1], 
    [2,  1]
    ])

b1 = np.array([-3, 0])

W2 = np.array([[2], [-1]])
b2 = np.array([1])

# 2. `Z1 = X @ W1 + b1`을 계산하여 출력하세요.
Z1 = X @ W1 + b1
print("== Z1 ==")
print(Z1)

# 3. `H = relu(Z1)`을 계산하여 출력하세요.
H = relu(Z1) 
print("\n == H ==")
print(H)


# 4. `Y = H @ W2 + b2`와 `Y.shape`를 출력하세요.
Y = H @ W2 + b2
print("\n == Y ==")
print(Y)
print("\n Y shape :", Y.shape)

== Z1 ==
[[ 1 -1]
 [ 4  2]]

 == H ==
[[1 0]
 [4 2]]

 == Y ==
[[3]
 [7]]

 Y shape : (2, 1)


#### 답변 작성란

**Q1. 고객 A의 은닉층 Z1과 ReLU 결과 H는 각각 무엇인가요?**

- Z1(은닉층) : [1 -1]  
-> 2x1 + 1x2 + (-3) , 2x-1 + 1x1 + 0  
-> 2 + 2 + (-3), -2 + 1 + 0  
-> [1 -1]  
- H (ReLU) : [1 0]  
-> max(0, 1), max(0, -1)  
-> 1은 0보다 크므로 1, -1은 0보다 작으므로 0  
-> [1 0]


**Q2. 고객 B의 은닉층 Z1과 ReLU 결과 H는 각각 무엇인가요?**

- Z1(은닉층) : [4  2]  
-> 1x1 + 3x2 + (-3), 1x-1 + 3x1 +0  
-> 1 + 6 + (-3), (-1) + 3 + 0
-> [4  2]

- H (ReLU) : [4  2]  
-> max(0, 4), max(0, 2)  
-> 4는 0보다 크므로 4, 2는 0보다 크므로 2  
-> [4  2]

**Q3. 고객 A와 B의 최종 점수는 각각 얼마인가요? H의 값을 사용해 계산하세요.**

- 고객 A : 3  
-> 1x2 + 0x-1 + 1  
-> 2 + 0 + 1  
-> 3  
- 고객 B : 7  
-> 4x2 + 2x-1 + 1  
-> 8 + -2 + 1  
-> 7  

**Q4. 최종 출력의 Shape는 무엇이고, 각 숫자는 무엇을 뜻하나요?**

- Y shape : (2, 1)  
- 2행 1열로 되어 있으며, 2는 고객의 수 (고객 A, B), 1은 고객 1명당 1개의 최종 점수를 뜻함

---

## 실습 마무리

1. 선형 변환과 아핀 변환의 계산은 어떻게 다른가요?
2. 가중치와 편향은 각각 언제 사용하는 숫자인가요?
3. ReLU는 [−3,−1,0,2,5]를 어떻게 바꾸나요?
4. 두 레이어의 계산에서 출력층에 전달하는 값은 무엇인가요?
5. 최종 출력의 Shape (3,1)는 무엇을 뜻하나요?
6. 주어진 가중치와 편향으로 출력만 계산하면 학습이 이루어지나요?

### 실습 마무리 답변 작성란

**1. 선형 변환과 아핀 변환의 계산은 어떻게 다른가요?**

- 답변: 선형 변환의 계산식 x @ W  
아핀 변환은 선형 변환의 결과에 편향을 더한 x @ W + b  
예시로 [0,6]에 편향 [1,-2]를 더하여 최종 [7,4]를 얻음

**2. 가중치와 편향은 각각 언제 사용하는 숫자인가요?**

- 답변: 가중치는 각 입력에 곱하는 숫자, 편향을 곱셈 결과들을 더한 뒤 추가로 더하는 숫자  
 4 x 2 + 2 x (-1) +3에서 가중치는 -2와 1이고, 편향은 3임  

**3. ReLU는 [−3,−1,0,2,5]를 어떻게 바꾸나요?**

- 답변:  [0,0,0,2,5]  
-> ReLU의 특성인 음수를 0으로 바꾸겠다는 기능이 있기 때문

**4. 두 레이어의 계산에서 출력층에 전달하는 값은 무엇인가요?**

- 답변: 은닉층의 가중합에 편향을 더한 Z1에 ReLU를 적용하여 만든 H  
순서는 Z1 = W1 @ b1, H = ReLU(Z1), Y + H  


**5. 최종 출력의 Shape (3,1)는 무엇을 뜻하나요?**

- 답변: 3행 1열, 고객 3명으로부터 여러 피쳐를 통해 하나의 값을 얻었다.  

**6. 주어진 가중치와 편향으로 출력만 계산하면 학습이 이루어지나요?**

- 답변: 아니오  
주어진 값으로 출력까지 계산하는 과정은 순전파

In [2]:
# 1장 3강

# 아핀변환 (-> 기울기 변화 없이 선 자체를 통째로 옮길 때 아핀변환을 씀)
# 선형 변환 결과에 일정한 벡터나 숫자 b를 더하는 계산
# 입력을 조합한 뒤에 기준 위치를 옮긴다.
# T(x) = xW + b
# 선형 결과 [6,6]에 b = [1,-2]를 더하면 [7,4]
# 왜 쓰냐? 화면속 점 [3,2]를 가로, 세로가 n배 늘어났고 옮긴다고 가정하에 아핀 변환을 사용할 수 있다.

# 뉴런
# 입력을 받아 출력 하나를 만드는 계산 단위
# ex) x1w1 + x2w2 + b
# 학생의 [출석,과제] 기록 [3,2]를 받아 중간값(은닉층 출력 결과)를 만드는 계산기가 있다.
# 가중치 [1,2]를 곱해 3 + 4 = 7을 만들고 편향 -4를 통해 3을 받는다.

# 레이어의 신경망
# 레이어 : 여러 뉴런을 모아 놓은 계산 단위
# 신경망 : 이런 계산들을 연결한 모델
# 학생의 기록 [3,2]를 단계별 계산을 하면
# 첫 단계의 뉴런 2개가 각각 3과 -3을 만들고 ReLU를 거쳐 3,0을 만들어내면
# 두번째 단계에서는 [3,0]을 이어받아서 3x2 + 0 x (-1)+1 = 7을 만든다
# 학생기록 -> 중간 숫자 -> 최종점수 <-이 각 단계가 레이어

# 활성화 함수
# 뉴런의 가중합에 편향을 더한 값을 다시 바꾸는 규칙
# ex) z = xw + b를 구하고 난 뒤 a=f(z)를 계산
# 학습실 이용 시간을 4시간 초과한 경우 금액을 부과하기 위해 따로 기록을 남긴다고 가정하면
# z = -4를 계산, 3시간을 이용했다면 z = -1로 추가적인 금액은 부과하지 않겠다.
# 이렇게 이미 계산된 식에 임의의 식을 더해서 그 기록과 규칙을 바꾸려고 하는게 활성화 함수다.

# 비선형 함수
# 비선형 함수는 선형 변환이 지켜야하는 덧셈 및 배수 관계를 모두 만족하지 않는 함수
# 추가 이용시간 y = ReLU(이용시간 -4)를 계산한다고 가정하면
# 3시간 이용자 수, 4시간 이용자 수는 추가 시간이 0인데
# 5시간, 6시간 이용자 수는 1시간, 2시간
# 이용 시간이 한 시간 늘어나도 기준 시간은 결과가 그대로이기 때문에 4시간 전까지 올바른 선형관계를 보이기 어렵다.

# 은닉층
# 입력과 최종 출력 사이에서 중간값을 만드는 계산 단계
# 은닉이라는 단어가 들어가 있지만 print()로 중간값들을 확인할 수는 있음
# 학생의 출석 및 과제 기록이 [3,2]로 최종 점수를 바로 만들지 않고 중간 계산을 거쳐서 검토한다고 가정해보면
# 두 뉴런으로 [3,-3]값을 만든 위 ReLU를 적용하여 [3,0]을 만든다 <- 이 과정을 은닉층이라고 부른다.
# 출력층(최종 산출)은 이 [3,0]을 조합하여 최종 점수를 만든다.
# 실제 신경망 계산에서는 입력을 여러 중간값으로 바꾸어 계산을 진행한다.

# 출력층
# 모델의 최종 결과를 만드는 계산 단계
# 앞 단계에서 학생의 중간값 H = [5,3]을 얻었다고 가정해보면 이 각 컬럼에 대한 점수들을 가지고 
# 실제 점수에 반영해보겠다.
# 10점 만점이라고 가정하고 가중치를 [2,-1], 편향을 1로 가정하면
# 5x2 + 3x-1 + 1 = 8점, 최종적으로 학생의 점수는 8점을 나타낼 수 있다.
# 출력층에 ReLU가 없다면 만약에 H = [0,3]이면 0 - 3 + 1 = -2로,
# 최종 출력값은 0이 아닌 -2로 계산될 수 있다.

# 순전파
# 입력부터 최종 출력까지 계산을 앞 방향으로 진행하는 과정
# 학생 기록 [3,2]를 모델에 넣고 계산 결과를 순전파로 따라가보면
# 1. 가중합과 편향을 계산 z1 = [3,-1]
# 2. ReLU를 적용하여 H = [3,0]을 만들고
# 3. 출력층에서 3x2 + 0x(-1) + 1 = 7을 산출한다.
# 1 -> 2 -> 3 이 과정을 순전파라고 부른다.


In [ ]:
# 2장 1강
# 분류 점수와 Logit
# 분류 점수(Logit)은 확률로 바꾸기 전에 모델이 계산한 숫자
# 메일이 스팸인지 분류하는 모델이 중간점수 z = 0.8이라는 점수를 냈다고 가정해보면
# 이 숫자 자체는 아직 80%를 의미하는 것이 아니라 그냥 그 점수 값을 의미한다.
# 이 숫자가 sigmoid라는 함수를 만나 0.69라는 확률로 계산이 될 수 있다.

# sigmoid 
# 점수 하나를 0과 1사이로 바꾸는 함수
# z = 0.8이면 -z = -0.8이고, e의 -0.8의 제곱은 약 0.449329면 여기에 1을 더한 1.449329로 1을 나누면
# 약 0.6899라는 결과를 얻을 수 있다.
# -> 각각의 점수를 바꿨을 때 이 시험을 통과할 확률을 구해주겠다

# softmax
# 한 대상의 여러 클래스 점수를 합이 1인 클래스별 확률로 바꾸는 함수
# ex) 문의 한 건을 [배송, 환불, 기타] 중 하나로 분류해야 한다면
# 모델이 예측한 결과 점수가 [2,1,0]이라고 가정해보면
# 최대값 2를 빼서 [0, -1, -2]로 만들고 지수값 약 [1, 0.36, 0.13]을 구한다.
# 각 지수 값의 합계인 약 1.5로 나누면 확률은 약 [0.66, 0.24, 0.09]고 <- 이 셋의 점수 합은 1이다.
# 이렇게 클래스별로 확률을 만든 뒤 가장 큰 확률을 가진 0.66(배송)을 선택하겠다.
# 문의 종류마다 각 배송, 환불, 기타의 확률값을 모두 볼 수 있다.

# Argmax
# 가장 큰 값이 있는 위치 번호를 찾는 연산
# ex) 문의의 [배송, 환불, 기타] 확률이 [0.2, 0.6, 0.2]라고 가정해보면
# 가장 큰 값은 0.6이고 위치는 두번째이다.
# 따라서 python에서 표현하면 np.argmax(p) = 1을 반환한다.
# 읽을 때에는 1번 환불이라고 읽으면 된다.
# 확률과 이름을 한 번에 같이 표현할 때 쓴다.

# 조건부 확률
# 특정 조건을 만족하는 경우만 모아서 보는 확률
# P(구매|장바구니 추가) = 장바구니 추가한 고객이라는 조건에서 구매할 확률, 세로줄 뒤가 조건
# 장바구니 추가 고객 6명 중 4명이 구매했다면 관측 구매 비율은 4 / 6 = 0.66 정도
# 단, 고객은 10명 기준이 아닌 장바구니를 추가한 고객 6명을 기준으로 계산했다는 점을 알고 있어야 한다.

# 베르누이 분포
# 결과가 두 가지일 때 한 번의 결과를 0 또는 1로 나타내는 확률 모형
# 계산식 Y = 1에서 p, y = 0에서 확률은 1-p
# kant 수업에서 한 학생이 과제를 제출했는지만 확인한다고 해보면 제출은 1, 미제출은 0으로 기록한다.
# 제출 확률을 p = 0.6으로 보면 제출에 0.6, 미제출에 0.4라는 점수를 부여한다.
# 이처럼 한가지 결과를 두가지로 표현하는 것이 베르누이 모형이다.

# 우도
# 이미 관측한 데이터를 고정해놓고, 파라미터에 따라 그 데이터에 부여되는 확률을 비교하는 함수
# y = [1, 0, 1, 1, 0], p = 6이면
# L(0.6) = 0.6 x 0.4 x 0.6 x 0.6 x 0.4 =0.03456
# 이 경우 L(p)는 p를 넣었을 때의 우도라고 읽을 수 있다.
# 위에 계산식을 말로 풀어보면
# 다섯개의 과제 제출 기록이 [1, 0, 1, 1, 0]이라고 하면 이 기록을 그대로 공통 제출 확률 두 개를 비교한다.
# p = 0.6이면 실제 기록에 준 확률의 곱은 0.6 x 0.4 x 0.6 x 0.6 x 0.4 =0.03456이고
# p = 0.8이면 실제 기록에 준 확률의 곱은 0.8 x 0.2 x 0.8 x 0.8 x 0.2 = 0.02048이다.
# 제출확률 80으로 설정을 해놓았다면 제출자에게는 유리하지만 미제출 두 명에게는 20%씩만 준다.
# -> 왜 쓸까? 어느 모델의 설정한 값이 지금 현재 기록에 더 잘 맞는지 숫자로 비교하고 싶다.

# probability(확률)와 Likelihood(우도)의 관점
# 두 개념은 확률식을 서로 다른 관전에서 바라본다.
# probability : 모델을 정해두고 가능한 결과를 살펴보는 방법
# Likelihood : 실제 출력된 결과를 보고 모델의 파라미터 값을 보겠다는 의미
# 확률 관점에서 p = 0.6일 때 라고 가정하면 이 0.6이라는 숫자값이 중요하다.
# 구매 확률이 60%야? 라는 질문을 던진다면
# 우도 관점에서는 실제 구매, 미구매 기록을 고정한 상태에서 L(0.6), L(0.8)을 비교하겠다는 의미
# 질문 - 수업 전에 이번 AI 기초 수학 과제를 제출할 확률을 0.6으로 설정했다면 원우가 이번 과제를 제출할 확률은? <- 확률 관점
# 수업이 끝난 뒤 [1, 0, 1, 1, 0]이라는 실제 기록을 확인한 뒤 p = 0.6과 p = 0.8 중 어느 설정이 이 기록에 더 큰 확률을 주는지? <- 우도 관점

# 파라미터 후보와 후보 비교
# 파라미터 후보 : 시험해 볼 파라미터의 값
# -> 각 후보에 같은 데이터를 넣어 우도를 구한 다음, 가장 큰 우도를 주는 후보를 선택
# p = [0.2, 0.4, 0.6, 0.8]의 우도는 순서대로 [0.005, 0.023, 0.034, 0.020]이다.
# 그렇다면 가장 큰 우도로 연결된 후보는 p = 0.6이다.
# -> [1, 0, 1, 1, 0]에 대한 제출 확률을 p = [0.2, 0.4, 0.6, 0.8] 산출하면 우도는 [0.005, 0.023, 0.034, 0.020]
# 따라서 제출 확률에 대한 우도값이 가장 큰 것은 p = 0.6이기 때문에 지금 [1,0,1,1,0] 기록에 더 잘맞는 설정값을 찾는 일(p = 0.6)
# p가 무조건 좋다고 좋은 것은 아님

# 최대우도추정과 MLE
# 관측 데이터의 우도가 가장 커지는 파라미터를 찾는 방법
# 공통 p를 사용하는 독립 베르누이 모형에서는 MLE = 구맥 = 구매수 / 전체고객수
# 구매3명, 전체 5명이면 3/5 = 0.6 <- 이 모형에서는 0~1사이에서 0.6이 우도를 가장 크게 만든다.
# ex) 예약 방문 기록이 [1,0,1,1,0]이라고 보면 방문 3명 미방문 2명이 모든 예약에 같은 방문확률 p를 사용하겠다로 가정
# 각 결과가 독립인 베르누이 모형에서는 우도를 가장 크게하는 p가 방문수 / 전체수인 3/5 = 0.6
# -> 0.6이 최대우도추정값이고 그때 전체 기록의 우도는 0.034정도
# -> 사람이 임의로 정한 확률 대신에 관측 자료결과에 근거하여 파라미터를 정하기 위해서 최대우도추정을 기준으로 사용한다.

# 인덱스와 Argmax
# 인덱스 : 배열의 위치 번호, np.argmax()는 가장 큰 값 자체가 아니라 그 값이 들어있는 위치 번호를 반환
# 우도 배열 [ 0.005, 0.023, 0.034, 0.020]이라고 가정해보면
# 가장 큰 우도는 0.034이며 순서는 세번째에 있다는 것을 알기 위함
# -> 가장 좋은 평가값을 빠르게 찾고 그 뒤 값을 만든 설정까지 빠르게 찾기 위함

# 로그
# 어떤 밑을 제곱하면 이 값이 되는가를 알려주는 연산자
# ex) 2를 3제곱하면 8이니까 밑이 2인 8의 로그는 3이다.
# 그렇다면 우도와 무슨 상관이냐
# 예약 세 건의 실제 결과에 확률이 0.8, 0.7, 0.7이라고 가정을 해보면
# 우도는 이 세 값을 곱한 0.392값을 의미한다.
# 이 우도의 자연로그는 ln(0.392)로 구할 수 있고 ln(0.8) + ln(0.7)로 얻을 수 있다.
# 즉, 로그의 특성을 이용하면 확률을 모두 곱하는 과정을 확률의 로그를 더하는 과정으로 바꿀 수가 있다.
# 기록이 매우 많아서 계속 곱해서 0에 한도 끝도 없이 가까워지면 구분하기 어려우니까
# 같은 비교지만 로그를 써서 안정적으로 계산하겠다는 의미.

# 로그 우도 (Log Likelihood)
# 우도에 자연로그를 적용한 값
# 각 관측 확률의 자연로그를 더해서도 같은 값을 얻는다.
# 같은 데이터에서는 큰 로그우도를 선호한다.
# q = [0.8, 0.7, 0.7]이면  L=0.392고, ln(L) = ln(0.8) + ln(0.7) + ln(0.7) = -0.936이다.
# 코드에서는 np.sum(np.log(q))를 사용할 수 있다.
# ex) 다른 모델이 같은 세 결과에 각각 0.5를 두면 우도는 0.125
# 로그 우도는 약 -2.0794값
# -0.9365 > -2.0794이기 때문에 첫 모델을 선택하려고 한다.
# 우도로 비교해도 0.392 > 0.125보다는 크기 때문에 같은 선택을 한다.

# 같은 데이터에서의 모델 비교
# 같은 고객의 같은 실제 결과에 대해 각 모델이 부여한 우도를 비교하겠다.
# 한 고객에게 더 좋은 확률을 준 모델이 다른 고객 또는 전체 데이터에서 항상 좋은 것은 아니다.
# 실제값 [1, 0, 1]에서 A의 관측확률 [0.8, 0.6, 0.6]은 우도 0.288
# B의 관측확률 [0.6, 0.8, 0.5]는 0.240 <- 이 두 모델을 비교했을 때는 A의 우도값이 높으니까 A모델을 선택하겠다.